In [ ]:
import pandas as pd
import numpy as np

# 1. Nạp tập dữ liệu sạch từ Phase 2
df = pd.read_csv("clean_telemetry_v2.csv")

# 2. Kiểm tra tổng quan kích thước và các trường thiếu dữ liệu
print(f"Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
print("\nThống kê các trường khuyết thiếu (Top 5):")
print(df.isna().sum().sort_values(ascending=False).head(5))
df.head()

Kích thước tập dữ liệu: 527 dòng, 25 cột

Thống kê các trường khuyết thiếu (Top 5):
notes              485
drone_id             0
operator_id          0
weight_ratio         0
overweight_flag      0
dtype: int64


,drone_id,application,drone_size,drone_model,manufacturer,propeller_count,max_carry_weight,actual_carry_weight,payload_type,payload_description,...,battery_remaining,gps_accuracy,wind_speed,obstacles_encountered,flight_status,regulatory_approval_id,notes,overweight_flag,weight_ratio,risk_score
0,D001,Package Delivery,Medium,FlyHigh 300,AeroCorp,4,5.0,2.5,Package,Small consumer goods,...,85,2.1,3.5,No,Completed,REG-2025-ABC-123,"Urban delivery, light package, good weather",0,0.500000,0.897508
1,D002,Infrastructure Inspection,Large,Inspecta X,SkyView Inc.,6,15.0,8.0,"Camera, Sensor","High-resolution camera, thermal sensor",...,68,1.8,6.2,Yes,Completed,REG-2025-DEF-456,"Bridge inspection, carried specialized equipment",0,0.533333,1.356176
2,D003,Agricultural Spraying,Large,CropMaster,AgriDrones,8,20.0,18.0,Liquid Tank,Pesticide solution,...,40,3.5,7.8,Yes,Completed,REG-2025-GHI-789,"Field spraying, full tank, moderate wind",0,0.900000,1.490365
3,D004,Aerial Photography,Small,SnapShot Mini,PhotoFly,4,1.0,0.5,Camera,Lightweight action camera,...,92,1.2,2.1,No,Completed,REG-2025-JKL-012,"Recreational use, clear skies",0,0.500000,1.076124
4,D005,Surveillance,Medium,Watcher Pro,SecureTech,4,7.0,3.0,"Camera, Sensor","Infrared camera, motion detection sensor",...,55,2.5,4.9,No,Completed,REG-2025-MNO-345,"Security patrol, nighttime operation",0,0.428571,1.229845


In [2]:
# 1. Tái cấu trúc biến mục tiêu từ đa phân loại sang nhị phân (Target Engineering)
df['flight_target'] = df['flight_status'].apply(lambda x: "Completed" if x == "Completed" else "Non-completed")

# 2. Phân rã ma trận đặc trưng đầu vào X và biến mục tiêu y
# Loại bỏ các cột định danh và các biến hậu biến cố (Post-flight features) để phục vụ Pre-dispatch DSS
drop_cols = [
    "flight_status", "flight_target", "drone_id", "operator_id", 
    "regulatory_approval_id", "flight_date",
    "flight_duration", "obstacles_encountered", "battery_remaining"
]
X = df.drop(columns=drop_cols)
y = df['flight_target']

print("✔ Phân tách đặc trưng đạt chuẩn Pre-dispatch DSS thành công!")
print(f"Danh sách {X.shape[1]} đặc trưng thời gian thực đưa vào mô hình:")
print(X.columns.tolist())
print("\nPhân phối nhãn mục tiêu nhị phân:")
print(y.value_counts())

✔ Phân tách đặc trưng đạt chuẩn Pre-dispatch DSS thành công!
Danh sách 17 đặc trưng thời gian thực đưa vào mô hình:
['application', 'drone_size', 'drone_model', 'manufacturer', 'propeller_count', 'max_carry_weight', 'actual_carry_weight', 'payload_type', 'payload_description', 'altitude', 'distance_flown', 'gps_accuracy', 'wind_speed', 'notes', 'overweight_flag', 'weight_ratio', 'risk_score']

Phân phối nhãn mục tiêu nhị phân:
Completed        522
Non-completed      5
Name: flight_target, dtype: int64


In [3]:
# 1. Tái cấu trúc biến mục tiêu từ đa phân loại sang nhị phân (Target Engineering)
df['flight_target'] = df['flight_status'].apply(lambda x: "Completed" if x == "Completed" else "Non-completed")

# 2. Phân rã ma trận đặc trưng đầu vào X và biến mục tiêu y
# Loại bỏ các cột định danh hệ thống không mang tính quy luật vận hành
drop_cols = ["flight_status", "flight_target", "drone_id", "operator_id", "regulatory_approval_id", "flight_date"]
X = df.drop(columns=drop_cols)
y = df['flight_target']

print("Phân phối nhãn mục tiêu nhị phân phục vụ DSS:")
print(y.value_counts())

Phân phối nhãn mục tiêu nhị phân phục vụ DSS:
Completed        522
Non-completed      5
Name: flight_target, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# Phân tách tập Train - Test theo tỷ lệ chuẩn 80/20 và giữ nguyên tỷ lệ phân phối nhãn
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Số lượng bản ghi huấn luyện (Train): {X_train.shape[0]}")
print(f"Số lượng bản ghi kiểm thử (Test): {X_test.shape[0]}")

Số lượng bản ghi huấn luyện (Train): 421
Số lượng bản ghi kiểm thử (Test): 106


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Phân loại tự động danh mục biến số thực và biến phân loại chữ
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Thiết lập đường ống xử lý tự động cho từng nhóm đặc trưng
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Gom hai nhánh xử lý vào bộ tiền xử lý trung tâm
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("✔ Khởi tạo Pipeline tiền xử lý dữ liệu tự động thành công!")

✔ Khởi tạo Pipeline tiền xử lý dữ liệu tự động thành công!


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Thiết lập mô hình đối chiếu 1: Logistic Regression với class_weight balanced
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Huấn luyện mô hình
log_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_log = log_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH LOGISTIC REGRESSION ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_log))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_log))

=== KẾT QUẢ MÔ HÌNH LOGISTIC REGRESSION ===
Accuracy: 1.0000

Ma trận nhầm lẫn (Confusion Matrix):
[[105   0]
 [  0   1]]

Báo cáo phân loại chi tiết (Classification Report):
               precision    recall  f1-score   support

    Completed       1.00      1.00      1.00       105
Non-completed       1.00      1.00      1.00         1

     accuracy                           1.00       106
    macro avg       1.00      1.00      1.00       106
 weighted avg       1.00      1.00      1.00       106



In [7]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

# Thiết lập mô hình đối chiếu 2: Random Forest Classifier với 200 cây quyết định
rf_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)), # Sử dụng clone để độc lập bộ nhớ với mô hình trước
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
])

# Huấn luyện mô hình
rf_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_rf = rf_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH RANDOM FOREST ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_rf))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_rf))

=== KẾT QUẢ MÔ HÌNH RANDOM FOREST ===
Accuracy: 0.9906

Ma trận nhầm lẫn (Confusion Matrix):
[[105   0]
 [  1   0]]

Báo cáo phân loại chi tiết (Classification Report):
               precision    recall  f1-score   support

    Completed       0.99      1.00      1.00       105
Non-completed       0.00      0.00      0.00         1

     accuracy                           0.99       106
    macro avg       0.50      0.50      0.50       106
 weighted avg       0.98      0.99      0.99       106



/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/envs/anaconda-2022.05-py39/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start,

In [8]:
# Tạo cấu trúc lưu trữ log siêu tham số hệ thống
hyperparams = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": None,
        "n_estimators": None
    },
    {
        "model": "Random Forest",
        "max_iter": None,
        "class_weight": "balanced",
        "random_state": 42,
        "n_estimators": 200
    }
])

# Xuất tệp tin lưu vết phục vụ kiểm toán mô hình
hyperparams.to_csv("hyperparameters_log.csv", index=False)
print("[SUCCESS] Đã dọn dẹp hệ thống và kết xuất file 'hyperparameters_log.csv' thành công!")

[SUCCESS] Đã dọn dẹp hệ thống và kết xuất file 'hyperparameters_log.csv' thành công!


In [9]:
import joblib
joblib.dump(log_model, "output/logistic_model.pkl")
joblib.dump(rf_model, "output/random_forest_model.pkl")

['output/random_forest_model.pkl']

In [10]:
hyperparams.to_csv("output/hyperparameters.csv", index=False)

In [11]:
import pandas as pd

results = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": 1.0},
    {"model": "Random Forest", "accuracy": 0.9905660377}
])

results.to_csv("output/model_results.csv", index=False)

In [12]:
pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_log
})
pred_df.to_csv("output/test_predictions.csv", index=False)